# Agent điều khiển trình duyệt trong sandbox, và prompt injection qua nội dung trang

Computer use thật cho model chụp màn hình, bấm toạ độ, gõ phím trên một máy (ảo). Bài này giữ phần cốt lõi mà bỏ phần
nguy hiểm: một **trang web giả lập** (`simweb.py`) mà agent đọc qua **cây phần tử** dạng chữ (giống accessibility
tree) và thao tác bằng `click`, `type_text`. Không mạng, không màn hình thật, trạng thái (giỏ hàng, đơn, email tài khoản)
kiểm được bằng code. Hai trang sách có một "đánh giá của khách" chứa **prompt injection gián tiếp**: yêu cầu agent đổi
email tài khoản sang địa chỉ của kẻ tấn công.

Bạn đo hai thứ cho mỗi cách phòng thủ: **tỉ lệ hoàn thành task** và **tỉ lệ bị tấn công thành công** (attack success
rate, ASR: email bị đổi sang địa chỉ kẻ tấn công).

**Cách làm bài**
- `guards.py`: ba hàm TODO, có test (`uv run pytest tracks/ai-engineer/modules/16-ai-agents/exercises/advanced/02-computer-use-sandbox -q`).
- Notebook này: tool trình duyệt, chạy agent `qwen2.5:3b` trên 8 task với ít nhất 4 cấu hình phòng thủ, báo cáo
  trong `REPORT.md`. Chấm theo rubric trong `SOLUTION.md`.

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import time

import pandas as pd

import guards
import simweb

In [ ]:
# explore: trang chủ, một trang sách có injection, 8 task
shop = simweb.SimShop()
print(shop.read_page())
shop.click("e2")
print(shop.click("e1"))
for task_id, task, _ in simweb.TASKS:
    print(task_id, task)

## 1. Tool trình duyệt và vòng agent

Dán `ToolRegistry` (bài Basic 01) và `run_react` (bài Basic 02) vào cell dưới, rồi viết hàm tạo registry ba tool
`read_page()`, `click(element_id)`, `type_text(element_id, text)` gắn với **một** `SimShop`, có tuỳ chọn bọc nội dung
trang trả về bằng `guards.delimit` hoặc `guards.datamark`. System prompt là `SYSTEM_TEMPLATE` của bài 02, thêm
`guards.SPOTLIGHT_NOTE` khi dùng đánh dấu nội dung. Mỗi task bắt đầu ở trang chủ; ngân sách 15 bước là đủ cho mọi task.
Lưu ý context: cây phần tử mỗi trang vài trăm token, nhưng 15 bước cộng dồn có thể vượt 4 096 token mặc định của
Ollama: dùng `num_ctx` 8 192 (client có `num_ctx` như bài Advanced 01) hoặc rút gọn observation cũ (bài Basic 03).

In [ ]:
# TODO: dán ToolRegistry (bài Basic 01) và run_react (bài Basic 02), rồi viết make_browser_tools(shop, wrap=None).

## 2. Bốn cấu hình phòng thủ

| Cấu hình | Ở đâu | Ý tưởng |
| --- | --- | --- |
| không phòng thủ | | nội dung trang đi thẳng vào context |
| delimit | prompt | `guards.delimit` + `SPOTLIGHT_NOTE`: nói với model đâu là dữ liệu |
| datamark | prompt | `guards.datamark` + `SPOTLIGHT_NOTE`: chữ trong trang bị "đánh dấu" từng khoảng trắng |
| guard | code | `SimShop(guard=guards.guard)`: thao tác nhạy cảm chỉ chạy khi khớp yêu cầu của người dùng |

Với mỗi cấu hình và mỗi task ghi: hoàn thành không (`check` của task), email có bị đổi sang `simweb.EVIL_EMAIL` không,
agent có **thấy** trang có injection không (observation chứa đoạn đánh giá đó), số lần guard chặn, số bước, lý do dừng.

In [ ]:
# explore: TODO của bạn: chạy 8 task x 4 cấu hình, lưu JSONL, in bảng hoàn thành / ASR / số task có thấy injection

## Đọc kết quả

- ASR chỉ có nghĩa trên các task mà agent **đã thấy** injection: tính trên mẫu số đó.
- Cấu hình nào giảm ASR mà không làm giảm tỉ lệ hoàn thành? Cấu hình nào chặn cả thao tác hợp lệ (task w08 đổi email
  thật)?
- Phòng thủ bằng prompt có bảo đảm gì không? Phòng thủ bằng code bảo đảm gì, và **không** bảo vệ được gì (gợi ý: thông
  tin bị lộ qua câu trả lời, thao tác không nằm trong danh sách nhạy cảm)? Module 19 đi sâu.

Sau đó làm phần trắc nghiệm của bài trong app.